# RescueEye — YOLO11s Training (Kaggle)

**University of Cebu – Banilad Campus · Capstone 2025–2026**

Trains `yolo11s` victim detection model on VisDrone aerial dataset.

**Before you start:**
- Settings (right sidebar) → Accelerator → **GPU T4 x2**
- Internet → **On**
- No Drive mount needed — outputs save to `/kaggle/working/` automatically
- After training, download `victim_best.onnx` from the Output tab

**Estimated time:** ~3–5 hours on T4 x2 (100 epochs, imgsz=1280)

## Step 1 — Install Dependencies

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()
print('\n✓ Dependencies ready')

## Step 2 — Set Paths

In [ ]:
import os

DATA_ROOT  = '/kaggle/working/rescueeye_data'
MODELS_DIR = '/kaggle/working/models'
VIS_DIR    = f'{DATA_ROOT}/raw/victim/visdrone'

os.makedirs(DATA_ROOT,  exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(VIS_DIR,    exist_ok=True)

print(f'Data root  : {DATA_ROOT}')
print(f'Models dir : {MODELS_DIR}')
print('\nOutputs will appear in the Output tab after training.')

## Step 3 — Download VisDrone Dataset

Downloads both **train** (6471 images) and **val** (548 images) splits from Ultralytics' mirror.

In [ ]:
import os, zipfile
from pathlib import Path

splits_to_download = [
    ('train', 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-train.zip', '/kaggle/working/visdrone_train.zip'),
    ('val',   'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip',   '/kaggle/working/visdrone_val.zip'),
]

for split_name, url, zip_path in splits_to_download:
    split_dir = f'{VIS_DIR}/{split_name}'
    if not os.path.exists(split_dir):
        print(f'Downloading VisDrone {split_name} ...')
        os.system(f'wget -q --show-progress -O {zip_path} {url}')
        os.makedirs(split_dir, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(split_dir)
        print(f'  Extracted to {split_dir}')
    else:
        print(f'VisDrone {split_name} already present.')

for split_name, _, _ in splits_to_download:
    imgs = list(Path(f'{VIS_DIR}/{split_name}').rglob('*.jpg'))
    print(f'  {split_name}: {len(imgs)} images')

## Step 4 — Prepare Dataset (VisDrone → YOLO format)

In [ ]:
import csv, shutil, yaml
from pathlib import Path
from PIL import Image

VICTIM_OUT  = Path(f'{DATA_ROOT}/victim')
PERSON_CATS = {1, 2}  # VisDrone: 1=pedestrian, 2=people

def visdrone_to_yolo(ann_path, img_w, img_h):
    lines = []
    with open(ann_path) as f:
        for row in csv.reader(f):
            if len(row) < 6: continue
            try: x,y,w,h,_,cat = int(row[0]),int(row[1]),int(row[2]),int(row[3]),row[4],int(row[5])
            except: continue
            if cat not in PERSON_CATS or w<=0 or h<=0: continue
            cx=(x+w/2)/img_w; cy=(y+h/2)/img_h
            lines.append(f'0 {cx:.6f} {cy:.6f} {w/img_w:.6f} {h/img_h:.6f}')
    return lines

for split in ('train', 'val'):
    split_raw = Path(f'{VIS_DIR}/{split}')
    img_candidates = list(split_raw.rglob('images'))
    img_root = img_candidates[0] if img_candidates else split_raw
    imgs = sorted(img_root.glob('*.jpg'))

    img_out = VICTIM_OUT / 'images' / split
    lbl_out = VICTIM_OUT / 'labels' / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    count = 0
    for img_path in imgs:
        ann_candidates = list(split_raw.rglob('annotations'))
        ann_root = ann_candidates[0] if ann_candidates else split_raw
        ann_path = ann_root / img_path.with_suffix('.txt').name
        w, h = Image.open(img_path).size
        lines = visdrone_to_yolo(ann_path, w, h) if ann_path.exists() else []
        shutil.copy2(img_path, img_out / img_path.name)
        (lbl_out / img_path.with_suffix('.txt').name).write_text('\n'.join(lines))
        count += 1
    print(f'  {split}: {count} images')

# Use val as test too
for folder in ('images', 'labels'):
    src = VICTIM_OUT / folder / 'val'
    dst = VICTIM_OUT / folder / 'test'
    if not dst.exists():
        shutil.copytree(src, dst)

victim_yaml = Path(f'{DATA_ROOT}/victim.yaml')
victim_yaml.write_text(yaml.dump({
    'path': str(VICTIM_OUT), 'train': 'images/train',
    'val': 'images/val', 'test': 'images/test', 'nc': 1, 'names': ['person']
}))
print(f'\n✓ victim.yaml written')
print(f'  Total images: {len(list((VICTIM_OUT/"images").rglob("*.jpg")))}')

## Step 5 — Train YOLO11s Victim Detection

Uses both T4 GPUs (`device='0,1'`), `imgsz=1280`, 100 epochs with early stopping (patience=15).

**Estimated time: ~3–5 hours**

In [ ]:
from ultralytics import YOLO
import time, shutil, json, csv as _csv
from pathlib import Path

VICTIM_RESULTS = f'{DATA_ROOT}/victim_results'
VICTIM_BEST    = f'{MODELS_DIR}/victim_best.pt'

model = YOLO('yolo11s.pt')
print('Training victim detection model (yolo11s, imgsz=1280, 100 epochs) ...')
print('Using both T4 GPUs.\n')

t0 = time.time()
results = model.train(
    data=str(victim_yaml),
    epochs=100,
    imgsz=1280,
    batch=4,
    patience=15,
    optimizer='AdamW', lr0=0.001, lrf=0.01,
    augment=True, mosaic=1.0, flipud=0.5, fliplr=0.5,
    scale=0.5,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    copy_paste=0.3,
    project=VICTIM_RESULTS, name='victim_train', exist_ok=True,
    device='0,1',
    workers=4,
)
elapsed = time.time() - t0

# DDP (multi-GPU) returns None — read metrics from results.csv instead
map50 = map5095 = prec = rec = 0.0
results_csv = Path(VICTIM_RESULTS) / 'victim_train' / 'results.csv'
if results_csv.exists():
    with open(results_csv) as f:
        rows = list(_csv.DictReader(f))
    if rows:
        last = {k.strip(): v.strip() for k, v in rows[-1].items()}
        map50   = float(last.get('metrics/mAP50(B)',    0))
        map5095 = float(last.get('metrics/mAP50-95(B)', 0))
        prec    = float(last.get('metrics/precision(B)', 0))
        rec     = float(last.get('metrics/recall(B)',    0))

print(f'\n─── Results ───────────────────────────────────')
print(f'  mAP@0.5       : {map50:.4f}  (target >= 0.70)')
print(f'  mAP@0.5:0.95  : {map5095:.4f}')
print(f'  Precision     : {prec:.4f}')
print(f'  Recall        : {rec:.4f}')
print(f'  Training time : {elapsed/60:.1f} min')

best_src = Path(VICTIM_RESULTS) / 'victim_train' / 'weights' / 'best.pt'
if best_src.exists():
    shutil.copy2(best_src, VICTIM_BEST)
    print(f'\n✓ Saved victim_best.pt → {VICTIM_BEST}')

meta = {'map50': round(map50,4), 'map50_95': round(map5095,4),
        'precision': round(prec,4), 'recall': round(rec,4),
        'training_time_min': round(elapsed/60,1),
        'trained_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
Path(f'{MODELS_DIR}/victim_meta.json').write_text(json.dumps(meta, indent=2))
print('✓ Saved victim_meta.json')

## Step 6 — Export to ONNX

In [ ]:
from ultralytics import YOLO
import shutil
from pathlib import Path

print('Exporting to ONNX (imgsz=1280) ...')
model = YOLO(VICTIM_BEST)
export_path = model.export(format='onnx', imgsz=1280, simplify=True)

onnx_dest = f'{MODELS_DIR}/victim_best.onnx'
try:
    shutil.copy2(export_path, onnx_dest)
except shutil.SameFileError:
    pass

print(f'\n✓ victim_best.onnx → {onnx_dest}')
print('\n── Download Instructions ──────────────────────')
print('1. Go to the Output tab (top-right of this notebook)')
print('2. Navigate to: rescueeye_data/models/')
print('3. Download: victim_best.onnx')
print('4. Place it in: api/models/victim_best.onnx')
print('5. Restart the FastAPI server')

## Done

After downloading `victim_best.onnx`:
1. Place it in `api/models/victim_best.onnx` on your local machine
2. Restart the API: `uvicorn main:app --reload --port 8000`
3. The dashboard should show the custom model version in the AI model info panel